# Notebook 4 — Evaluation Comparison

Run the fine-tuned model on the eval set and compare baseline vs fine-tuned accuracy.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from tqdm import tqdm
import torch
import matplotlib.pyplot as plt
from unsloth import FastLanguageModel

from src.dataset import load_stepgame, format_prompt
from src.eval import evaluate, save_results, load_predictions

In [ ]:
ADAPTER_PATH   = '../results/finetuned/lora_adapter'
EVAL_PATH      = '../data/eval/stepgame_eval.json'
OUT_PATH       = '../results/finetuned/predictions.json'
MAX_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 128

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

In [ ]:
examples = load_stepgame(EVAL_PATH)
predictions = []

for ex in tqdm(examples):
    prompt = format_prompt(ex['story'], ex['question'])
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    predictions.append({
        'answer': ex['answer'],
        'prediction': generated,
        'k': ex.get('k'),
    })

with open(OUT_PATH, 'w') as f:
    json.dump(predictions, f, indent=2)

ft_results = evaluate(predictions)
save_results(ft_results, '../results/finetuned/scores.json')

In [ ]:
import json

with open('../results/baseline/scores.json') as f:
    baseline = json.load(f)

print('Baseline accuracy:', baseline['accuracy'])
print('Fine-tuned accuracy:', ft_results['accuracy'])
print(f'Delta: +{ft_results["accuracy"] - baseline["accuracy"]:.3f}')

In [ ]:
# Per-hop comparison chart
k_vals = sorted(set(
    int(k.replace('accuracy_k', ''))
    for k in baseline if k.startswith('accuracy_k')
))

base_scores = [baseline.get(f'accuracy_k{k}', 0) for k in k_vals]
ft_scores   = [ft_results.get(f'accuracy_k{k}', 0) for k in k_vals]

x = range(len(k_vals))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - width/2 for i in x], base_scores, width, label='Baseline')
ax.bar([i + width/2 for i in x], ft_scores,   width, label='Fine-tuned')
ax.set_xticks(list(x))
ax.set_xticklabels([f'k={k}' for k in k_vals])
ax.set_ylabel('Accuracy')
ax.set_title('Spatial Reasoning Accuracy by Hop Level')
ax.legend()
plt.tight_layout()
plt.savefig('../results/comparison.png', dpi=150)
plt.show()